In [5]:
import os
from dotenv import load_dotenv

In [6]:
load_dotenv()

groq_key=os.getenv("GROQ_API_KEY")
langchain_tracing=os.getenv("LANGCHAIN_TRACING_V2")
langchain_project=os.getenv("LANGCHAIN_PROJECT")
langchain_key=os.getenv("LANGCHAIN_API_KEY")
huggingface_key=os.getenv("HF_TOKEN")

print("GROQ_KEY:", bool(groq_key))
print("LANGCHAIN_TRACING:", langchain_tracing)
print("LANGCHAIN_PROJECT:", bool(langchain_project))
print("LANGCHAIN_KEY:", bool(langchain_key))
print("HUGGINGFACE_KEY:", bool(huggingface_key))

GROQ_KEY: True
LANGCHAIN_TRACING: true
LANGCHAIN_PROJECT: True
LANGCHAIN_KEY: True
HUGGINGFACE_KEY: True


In [7]:
from langchain_groq import ChatGroq
model=ChatGroq(model="openai/gpt-oss-20b", groq_api_key=groq_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000173AEBB9CD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000173AED21110>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [8]:
from langchain_core.messages import HumanMessage, SystemMessage

message=[
    SystemMessage(content="Translate the following from english to German"),
    HumanMessage(content="Hello.. How are you?")
]
result=model.invoke(message)
result

AIMessage(content='Hallo… Wie geht es dir?', additional_kwargs={'reasoning_content': 'We need to translate "Hello.. How are you?" to German. Likely "Hallo... Wie geht es dir?" or "Hallo... Wie geht es Ihnen?" The user wrote "Hello.. How are you?" with two periods. It\'s informal. So translate to "Hallo… Wie geht es dir?" Use ellipsis. Provide translation.'}, response_metadata={'token_usage': {'completion_tokens': 85, 'prompt_tokens': 87, 'total_tokens': 172, 'completion_time': 0.092723435, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.008165475, 'prompt_tokens_details': None, 'queue_time': 0.2787038, 'total_time': 0.10088891}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_9340e7d14d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a06359-688d-7370-be4c-ef54ecf01958-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 87, 'output_tokens': 85, 'total_tokens': 1

In [9]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
parser.invoke(result)

'Hallo… Wie geht es dir?'

In [10]:
###  using LCEL- chaining the component
chain=model|parser
chain.invoke(message)

'Hallo… Wie geht es dir?'

In [13]:
### Prompt Template
from langchain_core.prompts import ChatPromptTemplate

generic_template="Translate the following into {language}"
prompt=ChatPromptTemplate.from_messages(
    [
        ("system", generic_template),
        ("user", "{text}")
    ]
)


In [15]:
result=prompt.invoke({
    "language":"French",
    "text":"hello"
    })

In [18]:
result

ChatPromptValue(messages=[SystemMessage(content='Translate the following into French', additional_kwargs={}, response_metadata={}), HumanMessage(content='hello', additional_kwargs={}, response_metadata={})])

In [19]:
result.to_messages()

[SystemMessage(content='Translate the following into French', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='hello', additional_kwargs={}, response_metadata={})]

In [20]:
chain=prompt|model|parser
chain.invoke({"language":"french","text":"hello how are you?"})

'Bonjour, comment vas-tu ?'